# 02 · `pip` & Virtual Environments in Depth

**Module 08 – Modules, Packaging & Dependency Management** · Estimated time: 60 min

> Based on 📘 *Virtual Environments & Package Management*, greatly extended. ⚠️ The 📗 bootcamp relies on **Anaconda/conda**; most Python developers now use **venv + pip**, **uv** (lesson 04) or **Poetry** (lesson 05).

### 🎯 What you will learn
- PyPI, packages, distributions and how `pip` installs them
- Every important `pip` command and option (version specifiers, extras, VCS/URL/local installs, editable mode, constraints, hashes)
- `requirements.txt` files done right
- Virtual environments: why, how `venv` works internally, all options, activation on every shell
- PEP 668 "externally managed environment" errors
- `pipx` for command-line tools
- A roadmap for the rest of this module

> 🧪 The commands below run in a **temporary sandbox folder** created by the next cell, so nothing touches your real environments. They need an internet connection (they download from PyPI).

In [1]:
import os, sys, tempfile, pathlib, subprocess
LAB = pathlib.Path(tempfile.mkdtemp(prefix="pip-lab-"))
os.environ["LAB"] = str(LAB)
os.environ["PY"] = sys.executable                 # this notebook's Python
os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
os.environ["PIP_NO_COLOR"] = "1"
print("Sandbox:", LAB)
print("Python :", sys.version.split()[0])

Sandbox: /var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/pip-lab-cwb8bqbn
Python : 3.15.0


## 1. Vocabulary

| Term | Meaning |
|---|---|
| **PyPI** | the Python Package Index, https://pypi.org (600k+ projects) |
| **Package** (import package) | a folder of modules you `import` |
| **Distribution** (dist) | an installable archive of a project: a **wheel** (`.whl`, pre-built) or **sdist** (`.tar.gz`, source) (lesson 06) |
| **Project name** vs **import name** | `pip install beautifulsoup4` but `import bs4`; `pip install pillow` but `import PIL` |
| **Installer** | a tool that downloads and installs distributions: `pip`, `uv` |
| **Environment** | a Python interpreter + its `site-packages` folder |
| **site-packages** | where installed packages live |

In [2]:
import site, sysconfig
print("site-packages:", sysconfig.get_paths()["purelib"])
import requests
print("requests is installed at:", requests.__file__)

site-packages: /private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f4ffaf20-c7ab-4c1c-ae64-d5055060b56d/scratchpad/venv315/lib/python3.15/site-packages
requests is installed at: /private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f4ffaf20-c7ab-4c1c-ae64-d5055060b56d/scratchpad/venv315/lib/python3.15/site-packages/requests/__init__.py


## 2. ✅ Always use `python -m pip`

`pip` (the bare command) might belong to a *different* Python than the `python` you run, a classic source of *"I installed it but get ModuleNotFoundError"*. `python -m pip` always uses the interpreter you name.

```bash
python -m pip --version          # shows which Python this pip belongs to
```
In Jupyter, use the `%pip` magic, which installs into the **running kernel**:
```python
%pip install requests
```

## 3. Create a virtual environment for our experiments

In [3]:
%%bash
cd "$LAB"
"$PY" -m venv .venv                  # create
ls .venv .venv/bin | head -20
cat .venv/pyvenv.cfg

.venv:
bin
include
lib
pyvenv.cfg

.venv/bin:
Activate.

ps1
activate
activate.csh
activate.fish
pip
pip3
pip3.15
pytho

n
python3
python3.15


home = /private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f4ffaf20-c7ab-4c1c-ae64-d505506

0b56d/scratchpad/uvpy/cpython-3.15-macos-aarch64-none/bin
include-system-site-packages = false
versi

on = 3.15.0
executable = /private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f4ffaf20-c7ab

-4c1c-ae64-d5055060b56d/scratchpad/uvpy/cpython-3.15.0-macos-aarch64-none/bin/python3.15
command = /

private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f4ffaf20-c7ab-4c1c-ae64-d5055060b56d/sc

ratchpad/venv315/bin/python -m venv /private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/pip-lab

-cwb8bqbn/.venv


### How a venv works
- `pyvenv.cfg` points to the **base** Python (`home = ...`)
- `.venv/bin/python` (Windows: `.venv\Scripts\python.exe`) is a link/copy of that interpreter
- When it starts, Python sees `pyvenv.cfg`, sets `sys.prefix` to the venv folder and uses the venv's **own `site-packages`**
- "Activating" only changes your shell's `PATH` (and prompt) so `python`/`pip` refer to the venv. You can also call `.venv/bin/python` directly without activating, which is what we do here

In [4]:
%%bash
cd "$LAB"
.venv/bin/python -c "import sys; print('prefix     :', sys.prefix); print('base prefix:', sys.base_prefix); print('in venv    :', sys.prefix != sys.base_prefix)"

prefix     : /private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/pip-lab-cwb8bqbn/.venv
base pr

efix: /private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f4ffaf20-c7ab-4c1c-ae64-d5055060

b56d/scratchpad/uvpy/cpython-3.15.0-macos-aarch64-none
in venv    : True


### Activating on each shell

| Shell | Command |
|---|---|
| macOS/Linux bash/zsh | `source .venv/bin/activate` |
| fish | `source .venv/bin/activate.fish` |
| Windows cmd | `.venv\Scripts\activate.bat` |
| Windows PowerShell | `.venv\Scripts\Activate.ps1` (may need `Set-ExecutionPolicy -Scope CurrentUser RemoteSigned`) |
| leave | `deactivate` |

### `venv` options

| Option | Effect |
|---|---|
| `--prompt NAME` | custom prompt prefix |
| `--upgrade-deps` | upgrade pip in the new venv |
| `--without-pip` | don't install pip (useful with uv) |
| `--system-site-packages` | also see the base Python's packages |
| `--clear` | delete and recreate an existing venv |
| `--copies` | copy the interpreter instead of symlinking |

✅ Conventions: one venv **per project**, named **`.venv`**, inside the project folder, **never committed** to Git (add `.venv/` to `.gitignore`). Recreate it any time from your dependency file.

## 4. Installing packages

In [5]:
%%bash
cd "$LAB"
PIP=".venv/bin/python -m pip"
$PIP install -q "requests==2.32.3"           # exact version
$PIP list
$PIP show requests | head -9

Package            Version


------------------ ----------
certifi            2026.10.19
charset-normalizer 3.5.2
idna           

    3.20
pip                26.2.1
requests           2.32.3
urllib3            2.8.0


Name: requests
Version: 2.32.3
Summary: Python HTTP for Humans.
Home-page: https://requests.readthed

ocs.io
Author: Kenneth Reitz
Author-email: me@kennethreitz.org
License: Apache-2.0
Location: /privat

e/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/pip-lab-cwb8bqbn/.venv/lib/python3.15/site-package

s
Requires: certifi, charset-normalizer, idna, urllib3


### Version specifiers (PEP 440 / PEP 508)

| Specifier | Meaning |
|---|---|
| `requests` | latest version |
| `requests==2.32.3` | exactly this version |
| `requests>=2.31` | at least |
| `requests>=2.31,<3` | range (most common for libraries) |
| `requests~=2.32.0` | "compatible release": `>=2.32.0, ==2.32.*` |
| `requests!=2.32.1` | anything except |
| `requests==2.*` | wildcard |
| `requests[socks]` | with the optional **extra** `socks` |
| `pywin32; sys_platform == "win32"` | **environment marker**: only on Windows |
| `tomli; python_version < "3.11"` | only on old Pythons |

> ⚠️ Quote specifiers in the shell (`"requests>=2.31"`); otherwise `>` is treated as redirection!

In [6]:
%%bash
cd "$LAB"
PIP=".venv/bin/python -m pip"
$PIP install -q --upgrade "requests>=2.32,<3"       # upgrade within a range
$PIP install -q "httpx[http2]"                      # an extra pulls in optional dependencies (h2)
$PIP list --format=freeze | grep -iE "^(requests|httpx|h2)=="
$PIP list --outdated --format=columns | head -5 || true

h2==4.4.1
httpx==0.28.1
requests==2.34.2


### Other install sources

```bash
pip install ./path/to/project                        # a local project folder (with pyproject.toml)
pip install -e ./path/to/project                     # EDITABLE: code changes apply without reinstalling
pip install ./dist/mypkg-1.0-py3-none-any.whl        # a wheel file
pip install "git+https://github.com/psf/requests.git@v2.32.3"   # from Git (tag/branch/commit)
pip install "mypkg @ https://example.com/mypkg-1.0.tar.gz"       # direct URL
pip install --index-url https://my.company/simple mypkg         # private index
pip install --extra-index-url https://download.pytorch.org/whl/cpu torch
pip install --user requests                          # per-user install (avoid; use venvs)
pip install --pre somepkg                            # allow pre-releases
pip install --no-deps somepkg                        # skip dependencies (rarely wise)
```

### Uninstalling and checking

In [7]:
%%bash
cd "$LAB"
PIP=".venv/bin/python -m pip"
$PIP uninstall -y -q httpx
$PIP check                      # verify installed packages have compatible dependencies
$PIP show -f requests | sed -n '1,3p'

No broken requirements found.


Name: requests
Version: 2.34.2
Summary: Python HTTP for Humans.


> ⚠️ `pip uninstall` removes only the named package, **not** the dependencies it pulled in (h2, httpcore... remain). That's one reason to rebuild environments from a lock/requirements file instead of uninstalling by hand.

## 5. `requirements.txt` done right

In [8]:
%%bash
cd "$LAB"
.venv/bin/python -m pip freeze > requirements.txt     # snapshot EVERYTHING installed, with exact versions
cat requirements.txt

anyio==4.15.1
certifi==2026.10.19
charset-normalizer==3.5.2
h11==0.16.0
h2==4.4.1
hpack==4.2.0
httpc

ore==1.0.9
hyperframe==6.1.0
idna==3.20
requests==2.34.2
urllib3==2.8.0


A requirements file can contain much more than `pkg==version`:

```text
# requirements.txt
-r base.txt                          # include another file
-c constraints.txt                   # constraints: cap versions without installing them
requests>=2.31,<3
pydantic[email]==2.14.1
-e ./libs/mylib                      # editable local package
git+https://github.com/user/repo.git@v1.2#egg=repo
--index-url https://pypi.org/simple
```

Common layout: `requirements.in` (your **direct** dependencies, loose) → **compiled** to `requirements.txt` (**all** dependencies pinned) with `pip-tools` (`pip-compile`) or `uv pip compile` (lesson 04).

### Reproducible & secure installs: hashes

```bash
pip install --require-hashes -r requirements.txt
```
```text
requests==2.32.3 \
    --hash=sha256:70761cfe03c773ceb22aa2f671b4757976145175cdfca038c02654d061d6dcc6
```
If any downloaded file doesn't match its hash, installation stops, which protects you from tampered packages.

### Recreate an environment from the file

In [9]:
%%bash
cd "$LAB"
"$PY" -m venv .venv2
.venv2/bin/python -m pip install -q -r requirements.txt
.venv2/bin/python -m pip list --format=freeze | grep -i requests

requests==2.34.2


## 6. Other useful pip commands

| Command | Purpose |
|---|---|
| `pip download pkg -d dir/` | download distributions without installing (offline installs) |
| `pip wheel pkg -w dir/` | build/download wheels for a package and its deps |
| `pip install --no-index --find-links dir/ pkg` | install from a local folder (air-gapped machines) |
| `pip cache dir` / `pip cache purge` | inspect / clear pip's cache |
| `pip config list` / `pip config set global.index-url URL` | persistent configuration (`pip.conf`/`pip.ini`) |
| `pip index versions pkg` | list available versions |
| `pip inspect` | JSON report of the environment |
| `pip install --dry-run --report report.json pkg` | resolve without installing |

In [10]:
%%bash
cd "$LAB"
.venv/bin/python -m pip index versions idna 2>/dev/null | head -2
.venv/bin/python -m pip download -q --no-deps -d wheels idna && ls wheels

idna (3.20)
Available versions: 3.20, 3.19, 3.18, 3.17, 3.16, 3.15, 3.14, 3.13, 3.12, 3.11, 3.10, 3.

9, 3.8, 3.7, 3.6, 3.5, 3.4, 3.3, 3.2, 3.1, 3.0, 2.10, 2.9, 2.8, 2.7, 2.6, 2.5, 2.4, 2.3, 2.2, 2.1, 2

.0, 1.1, 1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2


idna-3.20-py3-none-any.whl


## 7. PEP 668: "externally-managed-environment"

On many Linux distros, Homebrew Python and Debian/Ubuntu, `pip install` into the **system** Python fails:
```text
error: externally-managed-environment
× This environment is externally managed
```
This is deliberate: installing into the OS Python can break system tools. ✅ **Fix:** use a virtual environment (or `pipx`/`uv tool` for CLI apps). Don't reach for `--break-system-packages` unless you understand the consequences.

## 8. `pipx`: install command-line tools in isolation

Tools like `ruff`, `black`, `httpie`, `cookiecutter` are *applications*, not libraries for your project. `pipx` gives each its own hidden venv and puts the command on your `PATH`:

```bash
python -m pip install --user pipx && python -m pipx ensurepath
pipx install ruff          # ruff available everywhere, isolated
pipx run cowsay -t "hi"    # run once without installing
pipx upgrade-all
```
(`uv tool install` / `uvx` do the same, faster: lesson 04.)

## 9. Where this module goes next

| Lesson | Topic |
|---|---|
| 03 | **`pyproject.toml`**: the complete reference |
| 04 | **uv**: Python versions, projects, lock files, scripts, tools, workspaces, build & publish |
| 05 | **Poetry**: dependency management, groups, lock files, build & publish |
| 06 | **Building wheels & sdists** |
| 07 | **Publishing** with twine, uv and Poetry (PyPI/TestPyPI/private indexes) |
| 08 | Command-line apps with `argparse` |
| 09 | Choosing a tool: pip vs uv vs Poetry vs PDM vs Hatch vs conda |

## 📝 Summary
- Use `python -m pip` (or `%pip` in notebooks).
- One `.venv` per project; it's just a folder with its own `site-packages` and a `pyvenv.cfg`.
- Master version specifiers, extras and markers; quote them in the shell.
- `pip freeze > requirements.txt` snapshots; `-r` installs; hashes make installs tamper-proof.
- PEP 668 errors mean "use a venv"; use `pipx`/`uv tool` for CLI apps.

## 🏋️ Practice
1. Create a venv with a custom prompt `--prompt course`, activate it, and check `sys.prefix`.
2. Install `rich>=13,<15` and `httpx[http2]`; run `pip check`; freeze to `requirements.txt`.
3. Write a `requirements.in` with `requests` and `rich`, then compile it with `uv pip compile requirements.in -o requirements.txt` (after lesson 04).
4. Download wheels for `requests` and all dependencies into `wheels/`, then install offline with `--no-index --find-links wheels`.

<details><summary>Show solution for 4</summary>

```bash
python -m pip download requests -d wheels
python -m venv offline && offline/bin/python -m pip install --no-index --find-links wheels requests
```
</details>